In [2]:
import os
import sys
import getpass
import importlib
import tqdm
usrname = getpass.getuser()
sys.path.append('/Users/aghavamp/Desktop/Projects')
sys.path.append('/Users/aghavamp/Desktop/Projects/Functional_Fusion')
sys.path.append('/Users/aghavamp/Desktop/Projects/SUITPy')
sys.path.append('/Users/aghavamp/Desktop/Projects/AnatSearchlight')
sys.path.append(f'/Users/{usrname}/Desktop/Projects/PcmPy')

import AnatSearchlight.searchlight as sl
import scipy.io as sio
import rsatoolbox as rsa
from rsatoolbox.io import spm as spm_io
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
import surfAnalysisPy as surf
import SUITPy as suit
import nibabel as nb
from nibabel import cifti2
import nitools as nt
from matplotlib.cm import ScalarMappable
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from pathlib import Path
import seaborn as sns
import PcmPy as pcm
import Functional_Fusion.atlas_map as am
import Functional_Fusion.reliability as rel
import glob
import matplotlib.patches as patches
from nilearn.plotting.cm import _cmap_d as nilearn_cmaps

# SET PATHS:
baseDir = os.path.join('/Users', getpass.getuser(), 'Desktop', 'Projects', 'bimanual_wrist', 'data', 'fMRI')
bidsDir = 'BIDS'
anatomicalDir = 'anatomicals'
freesurferDir = 'surfaceFreesurfer'
surfacewbDir = 'surfaceWB' 
behavDir = 'behavioural'
regDir = 'ROI'
atlasDir = '/Volumes/diedrichsen_data$/data/Atlas_templates/fs_LR_32'
analysisDir = os.path.join(os.path.dirname(os.path.dirname(baseDir)), 'analysis')



There is the congruency effect and the bimanual-specific patterns that could derive the bimanual interaction term in our models. Here within each searchlight, we fit a full model with 'contra' + 'congruency' + 'bimanual-specific' terms. Then we will make the map corresponding to the 'congruency' and 'bimanual-specific' components. This allows us to visually see where exactly we have 'bimanual-specific' encoding vs. just congruency. This complements the full ROI analysis. 

### congruency component

In [ ]:
sn_list = [101, 104, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115]
glm = 1
hem = ['L', 'R']
sl_name = ['left_cortex', 'right_cortex']

def model_bimanual_interaction(data, **kwargs):
    Y_all = data[:-1,:] / np.sqrt(data[-1,:]) # univariate prewhitening
    part_vec = kwargs['partition_vec']
    cond_vec = kwargs['condition_vec']
    F = kwargs['F']
    G_incong = kwargs['G_incong']
    G_bimanual = kwargs['G_bimanual']
    
    # estimate the unimanual 6by6 G matrix from the given data:
    Guni_L = np.zeros((6,6))
    Guni_R = np.zeros((6,6))
    # unimanual conds are like lhand..., rhand... sorted in intrinsic order:
    rows_lhand = [j for j, c in enumerate(cond_vec) if c<6]
    rows_rhand = [j for j, c in enumerate(cond_vec) if (c>5 and c<12)]
    Guni_L,_ = pcm.est_G_crossval(Y_all[rows_lhand,:],
                                cond_vec[rows_lhand],
                                part_vec[rows_lhand],
                                X=pcm.matrix.indicator(part_vec[rows_lhand]))
    Guni_R,_ = pcm.est_G_crossval(Y_all[rows_rhand,:],
                                cond_vec[rows_rhand],
                                part_vec[rows_rhand],
                                X=pcm.matrix.indicator(part_vec[rows_rhand]))
    Guni = Guni_L/2 + Guni_R/2
    
    # make the contra+congruency+bimanual-specific model:
    G_contra = F @ Guni @ F.T # contra model with covariance structure
    G_contra = G_contra / np.trace(np.abs(G_contra)) # normalize by trace

    # make PCM dataset object:
    rows_bimanual = [j for j, c in enumerate(cond_vec) if c>11]
    Y_bimanual = Y_all[rows_bimanual,:]
    cond_vec_bimanual = cond_vec[rows_bimanual]
    part_vec_bimanual = part_vec[rows_bimanual]

    G_hat,_ = pcm.est_G_crossval(Y_bimanual,
                                cond_vec_bimanual,
                                part_vec_bimanual,
                                X=pcm.matrix.indicator(part_vec_bimanual))
    # D = pcm.Dataset(Y_bimanual, obs_descriptors={'cond_vec': cond_vec_bimanual, 'part_vec': part_vec_bimanual})
    
    # # fit CKA_individ:
    # T, theta = pcm.fit_CKA_individ(D, M, verbose=False)
    # weights = np.exp(theta[0]).flatten()

    # OLS:
    # mask = np.triu(np.ones((36, 36), dtype=bool)) # select only unique elements of G for OLS:
    X = np.vstack([
        G_contra.flatten(),
        G_incong.flatten(),
        G_bimanual.flatten(),
    ]).T
    # X = np.vstack([Gc_contra.flatten(), G_incong.flatten(), G_bimanual.flatten()]).T
    pinvX = np.linalg.pinv(X)
    W = pinvX @ G_hat.flatten()

    # OLS variances:
    ols_contra = W[0] * np.trace(G_contra)/36
    ols_congruency = W[1] * np.trace(G_incong)/36
    ols_bimanual = W[2] * np.trace(G_bimanual)/36

    d_contra = 2/(36-1) * ols_contra*36
    d_congruency = 2/(36-1) * ols_congruency*36
    d_bimanual = 2/(36-1) * ols_bimanual*36

    return d_congruency

# make parts of the models outside of the searclight loop for efficiency:
labels = ['flx_flx',    'flx_flxup',   'flx_extup',   'flx_ext',   'flx_extdn',   'flx_flxdn',
          'flxup_flx',  'flxup_flxup', 'flxup_extup', 'flxup_ext', 'flxup_extdn', 'flxup_flxdn',
          'extup_flx',  'extup_flxup', 'extup_extup', 'extup_ext', 'extup_extdn', 'extup_flxdn',
          'ext_flx',    'ext_flxup',   'ext_extup',   'ext_ext',   'ext_extdn',   'ext_flxdn',
          'extdn_flx',  'extdn_flxup', 'extdn_extup', 'extdn_ext', 'extdn_extdn', 'extdn_flxdn',
          'flxdn_flx',  'flxdn_flxup', 'flxdn_extup', 'flxdn_ext', 'flxdn_extdn', 'flxdn_flxdn']
F_contra = np.zeros((36,6))
cnd2idx = {'flx':0, 'flxup':1, 'extup':2, 'ext':3, 'extdn':4, 'flxdn':5}
for i in range(36):
    cond_pair = labels[i].split('_')
    cnd_contra = cond_pair[0]
    cnd_ipsi = cond_pair[1]
    idx_contra = cnd2idx[cnd_contra]
    idx_ipsi = cnd2idx[cnd_ipsi]
    F_contra[i, idx_contra] = 1
# center the features:
F_contra -= np.mean(F_contra, axis=0)
F = F_contra

# incongruency model:
ncond = 36
cov_incong = np.zeros((ncond,ncond)) #np.eye(ncond)
idx_incong = [1,2,4,5, 6,9,10,11, 12,15,16,17, 19,20,22,23, 24,25,26,27, 30,31,32,33]
for i in range(len(idx_incong)):
    for j in range(len(idx_incong)):
        if i != j:
            cov_incong[idx_incong[i], idx_incong[j]] = 1
F_incong = np.zeros((ncond,1))
F_incong[idx_incong,0] = 1
cov_incong = F_incong @ F_incong.T
G_incong = pcm.centering(ncond) @ cov_incong @ pcm.centering(ncond) # incongruency model
G_incong = G_incong / np.trace(G_incong) # normalize the overall variance

# bimanual-specific model:
I = np.eye(ncond)
G_bimanual = pcm.centering(ncond) @ I @ I.T @ pcm.centering(ncond) # non-orthogonalized interaction
G_bimanual = G_bimanual / np.trace(G_bimanual) # normalize the overall variance

for sn in sn_list:
    # try:
    # run marginal MVPA:
    # regressor info:
    reginfo = pd.read_table(os.path.join(baseDir, f'glm{glm}', f's{sn}', 'reginfo.tsv'))
    # get indices of the regressors of interest:
    # regressors_of_interest = reginfo[reginfo.name.str.contains('bi')].index.tolist()
    regressors_of_interest = reginfo.index.tolist()
    
    datafiles = [os.path.join(baseDir, f'glm{glm}', f's{sn}', f'beta_{i+1:04d}.nii') for i in regressors_of_interest]
    datafiles.append(os.path.join(baseDir, f'glm{glm}', f's{sn}', 'ResMS.nii'))
    partition_vec = reginfo.loc[regressors_of_interest, 'run'].values
    condition_vec = reginfo.loc[regressors_of_interest, 'name'].values

    for i, h in enumerate(hem):
        print(f'====================== Processing subject {sn}, hemisphere {h} ======================')
        S = sl.load(os.path.join(analysisDir,'searchlight',f'sl_{h}_s{sn}.h5'))

        # Make the condition_vec with a certain order required for model fitting:
        conditions = []
        # the unimanual conditions:
        lhand_conds = ['lhand:0',  'lhand:60',  'lhand:120',  'lhand:180',  'lhand:240',  'lhand:300']
        rhand_conds = ['rhand:180',  'rhand:120',  'rhand:60',  'rhand:0',  'rhand:300',  'rhand:240']
        conditions.extend(lhand_conds)
        conditions.extend(rhand_conds)

        # the bimanual conditions:
        angles_L = [0, 60, 120, 180, 240, 300]
        angles_R = [180, 120, 60, 0, 300, 240] # intrinsic order for right wrist directions
        if h == 'L':
                # right hand is contralateral:
                for angle_contra in angles_R:
                    for angle_ipsi in angles_L:
                        conditions.append(f'bi:{angle_ipsi}_{angle_contra}')
        if h == 'R':
                # left hand is contralateral:
                for angle_contra in angles_L:
                    for angle_ipsi in angles_R:
                        conditions.append(f'bi:{angle_contra}_{angle_ipsi}')

        # make cond_vec according to the sorting
        cond_vec = np.zeros_like(condition_vec, dtype=int)
        for cond in conditions:
            rows = [j for j, c in enumerate(condition_vec) if c.strip() == cond]
            cond_vec[rows] = conditions.index(cond)

        # run the searchlight:
        results = S.run_parallel(datafiles, model_bimanual_interaction, 
                                {'partition_vec': partition_vec, 'condition_vec': cond_vec, 'hem': h, 
                                'F': F, 'G_incong': G_incong, 'G_bimanual': G_bimanual})
        S.data_to_cifti(results, outfilename=os.path.join(baseDir, 'searchlight', f'glm{glm}_congruency_s{sn}_{h}_cortex.dscalar.nii'))



### bimanual-specific component

In [ ]:
sn_list = [101, 104, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115]
glm = 1
hem = ['L', 'R']
sl_name = ['left_cortex', 'right_cortex']

def model_bimanual_interaction(data, **kwargs):
    Y_all = data[:-1,:] / np.sqrt(data[-1,:]) # univariate prewhitening
    part_vec = kwargs['partition_vec']
    cond_vec = kwargs['condition_vec']
    F = kwargs['F']
    G_incong = kwargs['G_incong']
    G_bimanual = kwargs['G_bimanual']
    
    # estimate the unimanual 6by6 G matrix from the given data:
    Guni_L = np.zeros((6,6))
    Guni_R = np.zeros((6,6))
    # unimanual conds are like lhand..., rhand... sorted in intrinsic order:
    rows_lhand = [j for j, c in enumerate(cond_vec) if c<6]
    rows_rhand = [j for j, c in enumerate(cond_vec) if (c>5 and c<12)]
    Guni_L,_ = pcm.est_G_crossval(Y_all[rows_lhand,:],
                                cond_vec[rows_lhand],
                                part_vec[rows_lhand],
                                X=pcm.matrix.indicator(part_vec[rows_lhand]))
    Guni_R,_ = pcm.est_G_crossval(Y_all[rows_rhand,:],
                                cond_vec[rows_rhand],
                                part_vec[rows_rhand],
                                X=pcm.matrix.indicator(part_vec[rows_rhand]))
    Guni = Guni_L/2 + Guni_R/2
    
    # make the contra+congruency+bimanual-specific model:
    G_contra = F @ Guni @ F.T # contra model with covariance structure
    G_contra = G_contra / np.trace(np.abs(G_contra)) # normalize by trace

    # make PCM dataset object:
    rows_bimanual = [j for j, c in enumerate(cond_vec) if c>11]
    Y_bimanual = Y_all[rows_bimanual,:]
    cond_vec_bimanual = cond_vec[rows_bimanual]
    part_vec_bimanual = part_vec[rows_bimanual]

    G_hat,_ = pcm.est_G_crossval(Y_bimanual,
                                cond_vec_bimanual,
                                part_vec_bimanual,
                                X=pcm.matrix.indicator(part_vec_bimanual))
    # D = pcm.Dataset(Y_bimanual, obs_descriptors={'cond_vec': cond_vec_bimanual, 'part_vec': part_vec_bimanual})
    
    # # fit CKA_individ:
    # T, theta = pcm.fit_CKA_individ(D, M, verbose=False)
    # weights = np.exp(theta[0]).flatten()

    # OLS:
    # mask = np.triu(np.ones((36, 36), dtype=bool)) # select only unique elements of G for OLS:
    X = np.vstack([
        G_contra.flatten(),
        G_incong.flatten(),
        G_bimanual.flatten(),
    ]).T
    # X = np.vstack([Gc_contra.flatten(), G_incong.flatten(), G_bimanual.flatten()]).T
    pinvX = np.linalg.pinv(X)
    W = pinvX @ G_hat.flatten()

    # OLS variances:
    ols_contra = W[0] * np.trace(G_contra)/36
    ols_congruency = W[1] * np.trace(G_incong)/36
    ols_bimanual = W[2] * np.trace(G_bimanual)/36

    d_contra = 2/(36-1) * ols_contra*36
    d_congruency = 2/(36-1) * ols_congruency*36
    d_bimanual = 2/(36-1) * ols_bimanual*36

    return d_bimanual   

# make parts of the models outside of the searclight loop for efficiency:
labels = ['flx_flx',    'flx_flxup',   'flx_extup',   'flx_ext',   'flx_extdn',   'flx_flxdn',
          'flxup_flx',  'flxup_flxup', 'flxup_extup', 'flxup_ext', 'flxup_extdn', 'flxup_flxdn',
          'extup_flx',  'extup_flxup', 'extup_extup', 'extup_ext', 'extup_extdn', 'extup_flxdn',
          'ext_flx',    'ext_flxup',   'ext_extup',   'ext_ext',   'ext_extdn',   'ext_flxdn',
          'extdn_flx',  'extdn_flxup', 'extdn_extup', 'extdn_ext', 'extdn_extdn', 'extdn_flxdn',
          'flxdn_flx',  'flxdn_flxup', 'flxdn_extup', 'flxdn_ext', 'flxdn_extdn', 'flxdn_flxdn']
F_contra = np.zeros((36,6))
cnd2idx = {'flx':0, 'flxup':1, 'extup':2, 'ext':3, 'extdn':4, 'flxdn':5}
for i in range(36):
    cond_pair = labels[i].split('_')
    cnd_contra = cond_pair[0]
    cnd_ipsi = cond_pair[1]
    idx_contra = cnd2idx[cnd_contra]
    idx_ipsi = cnd2idx[cnd_ipsi]
    F_contra[i, idx_contra] = 1
# center the features:
F_contra -= np.mean(F_contra, axis=0)
F = F_contra

# incongruency model:
ncond = 36
cov_incong = np.zeros((ncond,ncond)) #np.eye(ncond)
idx_incong = [1,2,4,5, 6,9,10,11, 12,15,16,17, 19,20,22,23, 24,25,26,27, 30,31,32,33]
for i in range(len(idx_incong)):
    for j in range(len(idx_incong)):
        if i != j:
            cov_incong[idx_incong[i], idx_incong[j]] = 1
F_incong = np.zeros((ncond,1))
F_incong[idx_incong,0] = 1
cov_incong = F_incong @ F_incong.T
G_incong = pcm.centering(ncond) @ cov_incong @ pcm.centering(ncond) # incongruency model
G_incong = G_incong / np.trace(G_incong) # normalize the overall variance

# bimanual-specific model:
I = np.eye(ncond)
G_bimanual = pcm.centering(ncond) @ I @ I.T @ pcm.centering(ncond) # non-orthogonalized interaction
G_bimanual = G_bimanual / np.trace(G_bimanual) # normalize the overall variance

for sn in sn_list:
    # try:
    # run marginal MVPA:
    # regressor info:
    reginfo = pd.read_table(os.path.join(baseDir, f'glm{glm}', f's{sn}', 'reginfo.tsv'))
    # get indices of the regressors of interest:
    # regressors_of_interest = reginfo[reginfo.name.str.contains('bi')].index.tolist()
    regressors_of_interest = reginfo.index.tolist()
    
    datafiles = [os.path.join(baseDir, f'glm{glm}', f's{sn}', f'beta_{i+1:04d}.nii') for i in regressors_of_interest]
    datafiles.append(os.path.join(baseDir, f'glm{glm}', f's{sn}', 'ResMS.nii'))
    partition_vec = reginfo.loc[regressors_of_interest, 'run'].values
    condition_vec = reginfo.loc[regressors_of_interest, 'name'].values

    for i, h in enumerate(hem):
        print(f'====================== Processing subject {sn}, hemisphere {h} ======================')
        S = sl.load(os.path.join(analysisDir,'searchlight',f'sl_{h}_s{sn}.h5'))

        # Make the condition_vec with a certain order required for model fitting:
        conditions = []
        # the unimanual conditions:
        lhand_conds = ['lhand:0',  'lhand:60',  'lhand:120',  'lhand:180',  'lhand:240',  'lhand:300']
        rhand_conds = ['rhand:180',  'rhand:120',  'rhand:60',  'rhand:0',  'rhand:300',  'rhand:240']
        conditions.extend(lhand_conds)
        conditions.extend(rhand_conds)

        # the bimanual conditions:
        angles_L = [0, 60, 120, 180, 240, 300]
        angles_R = [180, 120, 60, 0, 300, 240] # intrinsic order for right wrist directions
        if h == 'L':
                # right hand is contralateral:
                for angle_contra in angles_R:
                    for angle_ipsi in angles_L:
                        conditions.append(f'bi:{angle_ipsi}_{angle_contra}')
        if h == 'R':
                # left hand is contralateral:
                for angle_contra in angles_L:
                    for angle_ipsi in angles_R:
                        conditions.append(f'bi:{angle_contra}_{angle_ipsi}')

        # make cond_vec according to the sorting
        cond_vec = np.zeros_like(condition_vec, dtype=int)
        for cond in conditions:
            rows = [j for j, c in enumerate(condition_vec) if c.strip() == cond]
            cond_vec[rows] = conditions.index(cond)

        # run the searchlight:
        results = S.run_parallel(datafiles, model_bimanual_interaction, 
                                {'partition_vec': partition_vec, 'condition_vec': cond_vec, 'hem': h, 
                                'F': F, 'G_incong': G_incong, 'G_bimanual': G_bimanual})
        S.data_to_cifti(results, outfilename=os.path.join(baseDir, 'searchlight', f'glm{glm}_bimanual_specific_s{sn}_{h}_cortex.dscalar.nii'))

